# Task 1: K-Means Customer Segmentation

**Dataset:** Mall Customers (`Mall_Customers.csv`)  
**Goal:** Segment customers using K-Means clustering, determine the optimal number of clusters via the Elbow method and Silhouette analysis, visualise results, and derive business insights.

---
## Section 1 — Data Loading & Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('datasets/Mall_Customers.csv')

print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Mall Customers — Feature Distributions', fontsize=14, fontweight='bold')

sns.countplot(data=df, x='Gender', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Gender Distribution')

sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Age Distribution')

sns.histplot(df['Annual Income (k$)'], bins=20, kde=True, ax=axes[1, 0], color='darkorange')
axes[1, 0].set_title('Annual Income (k$) Distribution')

sns.histplot(df['Spending Score (1-100)'], bins=20, kde=True, ax=axes[1, 1], color='forestgreen')
axes[1, 1].set_title('Spending Score Distribution')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Income & Spending Score by Gender', fontsize=13, fontweight='bold')

sns.boxplot(data=df, x='Gender', y='Annual Income (k$)', ax=axes[0], palette='Set2')
axes[0].set_title('Annual Income by Gender')

sns.boxplot(data=df, x='Gender', y='Spending Score (1-100)', ax=axes[1], palette='Set2')
axes[1].set_title('Spending Score by Gender')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Gender',
    palette='Set1',
    alpha=0.7,
    s=60
)
plt.title('Annual Income vs Spending Score (raw data)')
plt.tight_layout()
plt.show()

**EDA observations:**
- 200 customers, no missing values.
- Gender is roughly balanced (slightly more Female records).
- Age ranges from 18 to 70; bimodal income distribution hints at distinct customer groups.
- The scatter plot of Annual Income vs Spending Score already suggests 4–5 natural clusters.

---
## Section 2 — Preprocessing

In [ ]:
df_clean = df.copy()

# Drop the identifier — carries no clustering signal
df_clean.drop(columns=['CustomerID'], inplace=True)

# Encode Gender as binary for use in pair plots / extended analysis
df_clean['Gender_enc'] = (df_clean['Gender'] == 'Male').astype(int)

# Numeric feature columns
num_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

# Primary 2-feature set (Income + Score) — most discriminative for K-Means
X2 = df_clean[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Full 3-feature set (Age + Income + Score) — used for pair plots
X3 = df_clean[num_cols].values

scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)

scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)

print('X2 (Income + Score) shape:', X2_scaled.shape)
print('X3 (Age + Income + Score) shape:', X3_scaled.shape)

**Preprocessing decisions:**
- `CustomerID` is dropped — it is a unique row identifier with no clustering signal.
- `Gender` is label-encoded (`Male=1, Female=0`) for pair-plot colouring; it is *not* used as a clustering feature because K-Means assumes Euclidean distance, which is not meaningful for binary nominal variables.
- `StandardScaler` ensures all numeric features have mean=0 and std=1, preventing larger-magnitude features (e.g. income in k\$) from dominating the distance calculation.
- Primary clustering is performed on **Annual Income** and **Spending Score** — the two features that visually show the clearest natural groupings.

---
## Section 3 — Optimal K Selection: Elbow Method & Silhouette Analysis

In [ ]:
# ── Elbow Method ────────────────────────────────────────────────────────────
K_range = range(1, 11)
inertias = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    km.fit(X2_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_range), inertias, marker='o', color='steelblue', linewidth=2)
ax.axvline(x=5, color='crimson', linestyle='--', linewidth=1.5, label='Elbow at k=5')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Within-Cluster Sum of Squares (Inertia)')
ax.set_title('Elbow Method — Optimal K Selection')
ax.legend()
plt.tight_layout()
plt.show()

print('Inertias:', dict(zip(K_range, [round(v, 1) for v in inertias])))

In [ ]:
# ── Silhouette Score Plot ────────────────────────────────────────────────────
sil_scores = []
K_sil = range(2, 11)

for k in K_sil:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X2_scaled)
    sil_scores.append(silhouette_score(X2_scaled, labels))

best_k = K_sil[np.argmax(sil_scores)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(K_sil), sil_scores, color='steelblue', edgecolor='white')
ax.axvline(x=best_k, color='crimson', linestyle='--', linewidth=1.5,
           label=f'Best k={best_k} (score={max(sil_scores):.3f})')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Average Silhouette Score')
ax.set_title('Silhouette Analysis — Optimal K Selection')
ax.legend()
plt.tight_layout()
plt.show()

print('Silhouette scores:', dict(zip(K_sil, [round(s, 4) for s in sil_scores])))
print(f'Best k by silhouette: {best_k}')

In [ ]:
# ── Silhouette Diagrams for k=4, 5, 6 ───────────────────────────────────────
candidates = [4, 5, 6]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Silhouette Diagrams for Candidate Values of k', fontsize=13, fontweight='bold')

for ax, k in zip(axes, candidates):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X2_scaled)
    avg_score = silhouette_score(X2_scaled, labels)
    sample_scores = silhouette_samples(X2_scaled, labels)

    y_lower = 10
    colors = cm.tab10(np.linspace(0, 1, k))
    for i in range(k):
        cluster_vals = np.sort(sample_scores[labels == i])
        size = cluster_vals.shape[0]
        y_upper = y_lower + size
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_vals,
                         alpha=0.7, color=colors[i])
        ax.text(-0.05, y_lower + 0.5 * size, str(i), fontsize=8)
        y_lower = y_upper + 10

    ax.axvline(x=avg_score, color='crimson', linestyle='--', linewidth=1.5)
    ax.set_title(f'k={k}  (avg={avg_score:.3f})')
    ax.set_xlabel('Silhouette coefficient')
    ax.set_xlim([-0.2, 1])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

**Optimal K Justification:**

Both methods converge on **k = 5**:

| Method | Evidence |
|---|---|
| Elbow | The inertia curve bends sharply at k=5; gains beyond 5 are marginal |
| Silhouette | k=5 yields the highest average silhouette score; all 5 clusters are above the average line in the diagram |

The silhouette diagrams also confirm that k=4 and k=6 produce clusters of unequal width (one cluster is notably thin), whereas k=5 shows balanced, well-separated clusters.

---
## Section 4 — K-Means Implementation

In [ ]:
OPTIMAL_K = 5

kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE
)
kmeans.fit(X2_scaled)

df_clean['Cluster'] = kmeans.labels_

print('Cluster counts:')
print(df_clean['Cluster'].value_counts().sort_index())
print(f'\nInertia: {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X2_scaled, kmeans.labels_):.4f}')

In [ ]:
cluster_summary = df_clean.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)
cluster_summary.columns = ['Mean Age', 'Mean Income (k$)', 'Mean Spending Score']
print(cluster_summary)

---
## Section 5 — Visualisations

In [ ]:
# ── 2D Scatter: Income vs Spending Score ────────────────────────────────────
palette = sns.color_palette('tab10', OPTIMAL_K)

# Inverse-transform centroids to original scale for annotation
centroids_original = scaler2.inverse_transform(kmeans.cluster_centers_)

plt.figure(figsize=(9, 6))
for cluster_id in range(OPTIMAL_K):
    mask = df_clean['Cluster'] == cluster_id
    plt.scatter(
        df_clean.loc[mask, 'Annual Income (k$)'],
        df_clean.loc[mask, 'Spending Score (1-100)'],
        color=palette[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.75,
        s=60
    )

plt.scatter(
    centroids_original[:, 0],
    centroids_original[:, 1],
    color='black',
    marker='X',
    s=200,
    zorder=10,
    label='Centroids'
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1–100)')
plt.title('K-Means Clustering (k=5): Annual Income vs Spending Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Pair Plot: Age, Income, Spending Score ───────────────────────────────────
pair_df = df_clean[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Cluster']].copy()
pair_df['Cluster'] = pair_df['Cluster'].astype(str)

g = sns.pairplot(
    pair_df,
    hue='Cluster',
    palette='tab10',
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 's': 40}
)
g.fig.suptitle('Pair Plot of Customer Features Coloured by Cluster', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cluster Profiles: Bar chart of mean values ───────────────────────────────
cluster_means = df_clean.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()

ax = cluster_means.plot(
    kind='bar',
    figsize=(10, 5),
    colormap='tab10',
    edgecolor='white'
)
ax.set_title('Mean Feature Values per Cluster', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## Section 6 — Business Insights

In [ ]:
# Cluster summary statistics (original, unscaled values)
summary = df_clean.groupby('Cluster')[[
    'Age', 'Annual Income (k$)', 'Spending Score (1-100)'
]].mean().round(1)

print('Cluster Summary (unscaled means):')
print(summary.to_string())
print('\nGender distribution per cluster:')
print(df_clean.groupby('Cluster')['Gender'].value_counts().unstack(fill_value=0))

In [ ]:
# ── Assign human-readable segment names ─────────────────────────────────────
# Strategy:
#   1. The cluster whose centroid is closest to the overall dataset mean is
#      labelled "Average Customers" (it sits in the middle of the feature space).
#   2. The remaining four clusters are labelled by Income × Score quadrant:
#      High Income / High Score  → Premium Spenders
#      Low Income  / High Score  → Aspirational Spenders
#      High Income / Low Score   → Conservative Savers
#      Low Income  / Low Score   → Budget Cautious

centroid_df = pd.DataFrame(
    centroids_original,
    columns=['Annual Income (k$)', 'Spending Score (1-100)']
)

overall_mean = np.array([
    df_clean['Annual Income (k$)'].mean(),
    df_clean['Spending Score (1-100)'].mean()
])

# Euclidean distance from each centroid to the overall mean
centroid_df['dist_to_mean'] = np.linalg.norm(
    centroid_df[['Annual Income (k$)', 'Spending Score (1-100)']].values - overall_mean,
    axis=1
)
avg_cluster = centroid_df['dist_to_mean'].idxmin()

income_median = df_clean['Annual Income (k$)'].median()
score_median  = df_clean['Spending Score (1-100)'].median()

def assign_segment(cluster_id, inc, sc):
    if cluster_id == avg_cluster:
        return 'Average Customers'
    if inc >= income_median and sc >= score_median:
        return 'Premium Spenders'
    elif inc < income_median and sc >= score_median:
        return 'Aspirational Spenders'
    elif inc >= income_median and sc < score_median:
        return 'Conservative Savers'
    else:
        return 'Budget Cautious'

cluster_to_segment = {
    i: assign_segment(i, centroid_df.loc[i, 'Annual Income (k$)'], centroid_df.loc[i, 'Spending Score (1-100)'])
    for i in range(OPTIMAL_K)
}

df_clean['Segment'] = df_clean['Cluster'].map(cluster_to_segment)

print('Cluster → Segment mapping:')
for k, v in sorted(cluster_to_segment.items()):
    print(f'  Cluster {k}: {v}')

In [ ]:
# ── Labelled Scatter Plot ────────────────────────────────────────────────────
segment_palette = {
    'Premium Spenders':       '#2ca02c',
    'Aspirational Spenders':  '#ff7f0e',
    'Conservative Savers':    '#1f77b4',
    'Budget Cautious':        '#d62728',
    'Average Customers':      '#9467bd',
}

plt.figure(figsize=(10, 6))
for segment, color in segment_palette.items():
    mask = df_clean['Segment'] == segment
    if mask.any():
        plt.scatter(
            df_clean.loc[mask, 'Annual Income (k$)'],
            df_clean.loc[mask, 'Spending Score (1-100)'],
            color=color,
            label=segment,
            alpha=0.75,
            s=65
        )

plt.scatter(
    centroids_original[:, 0], centroids_original[:, 1],
    color='black', marker='X', s=220, zorder=10, label='Centroids'
)
plt.xlabel('Annual Income (k$)', fontsize=11)
plt.ylabel('Spending Score (1–100)', fontsize=11)
plt.title('Customer Segments — K-Means (k=5)', fontsize=13, fontweight='bold')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── Segment summary table ────────────────────────────────────────────────────
seg_summary = df_clean.groupby('Segment').agg(
    Count=('Cluster', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income=('Annual Income (k$)', 'mean'),
    Avg_Score=('Spending Score (1-100)', 'mean')
).round(1).sort_values('Avg_Income', ascending=False)

print(seg_summary.to_string())

### Business Insights per Segment

| Segment | Income | Spending Score | Key Characteristics | Business Strategy |
|---|---|---|---|---|
| **Premium Spenders** | High | High | Likely younger–middle-aged, disposable income and willingness to spend | VIP loyalty programmes, premium product launches, exclusive early access |
| **Conservative Savers** | High | Low | Earn well but spend cautiously; older on average | Value-focused offers, long-term investment products, trust-building campaigns |
| **Aspirational Spenders** | Low | High | Young shoppers who spend beyond their income level | Instalment/BNPL options, aspirational brand collaborations, flash sales |
| **Budget Cautious** | Low | Low | Price-sensitive; likely comparison shoppers | Discount promotions, bundle deals, generic/own-brand alternatives |
| **Average Customers** | Mid | Mid | The broad middle market; not strongly differentiated | General promotions, cross-selling, push-notification campaigns |

**Key takeaway:** The most commercially valuable action is to focus retention efforts on **Premium Spenders** and **Aspirational Spenders**, while deploying targeted campaigns to nudge **Conservative Savers** into higher spending using value-oriented messaging.

---
## Section 7 — Question Responses

### Q1. Why is determining the appropriate number of clusters crucial in customer segmentation? Compare the Elbow method and Silhouette analysis. When would you prefer one over the other?

Choosing the right number of clusters (k) is fundamental because it directly determines the granularity and usefulness of the resulting segments. Too few clusters merge genuinely distinct customer groups, hiding actionable differences; too many clusters over-fragment the data into groups so small or similar that targeting them separately yields no incremental business value. An inappropriate k can also mislead decision-making — a business may allocate marketing budget to a fictitious segment that is actually an artefact of the algorithm rather than a real behavioural group.

**Elbow Method**  
The elbow method plots within-cluster sum of squares (WCSS / inertia) against k. As k increases, inertia always decreases, but the rate of decrease slows. The "elbow" — the point where adding another cluster yields diminishing returns — is chosen as the optimal k.

- *Strengths:* Intuitive, computationally cheap, easy to explain to non-technical stakeholders.
- *Weaknesses:* The elbow is often ambiguous (a gradual curve rather than a sharp bend), and it only measures compactness — it says nothing about how well-separated clusters are from each other.
- *Prefer when:* You need a quick initial estimate, the dataset is large (silhouette is O(n²)), or you need a result that is easy to justify to business partners.

**Silhouette Analysis**  
The silhouette score for a sample measures how similar it is to its own cluster versus the nearest neighbouring cluster, ranging from −1 (misclassified) to +1 (well-matched). The average score across all samples is maximised at the optimal k; silhouette diagrams additionally reveal whether individual clusters are consistent.

- *Strengths:* Measures both cohesion and separation simultaneously; silhouette diagrams expose unbalanced or overlapping clusters that a summary inertia curve cannot reveal.
- *Weaknesses:* More computationally expensive (O(n²) distance calculations); can favour compact spherical clusters, inheriting some of K-Means' own assumptions.
- *Prefer when:* You need a rigorous, quantitative comparison across multiple k values; when you suspect the elbow is ambiguous; or when cluster quality (not just compactness) matters.

In practice, the two methods are **complementary**: use the elbow to narrow the search range and silhouette analysis to confirm and refine the final choice. In this task, both methods agreed on k=5, which gives high confidence in the segmentation.

---

### Q2. What are the key limitations of K-Means clustering in customer segmentation? Discuss how poor initialisation, sensitivity to outliers, and assumptions about cluster shapes can affect results. How can these limitations be addressed?

**1. Poor Initialisation**  
K-Means initialises centroids randomly, meaning different runs can converge to different local minima. A poor starting configuration leads to suboptimal clusters that are inconsistent across runs.

*Impact:* Two separate analyses of the same data could produce different segment definitions, making it unreliable in production.  
*Mitigation:* Use **K-Means++** initialisation (the default in scikit-learn), which spreads initial centroids probabilistically to reduce the risk of bad starts. Also run with `n_init=10` (multiple restarts) and select the run with the lowest inertia.

**2. Sensitivity to Outliers**  
K-Means minimises squared Euclidean distance; outliers (e.g. a customer with an annual income of \$300k) exert a disproportionate pull on centroids and can create a cluster that is essentially a "catch-all" for anomalies rather than a meaningful segment.

*Impact:* Centroids shift toward outliers, distorting the shape of otherwise well-defined clusters and degrading silhouette scores.  
*Mitigation:* Remove or cap outliers before clustering; use **RobustScaler** instead of StandardScaler (scales by median and IQR, making it less sensitive to extreme values); or apply **K-Medoids** (PAM), which uses actual data points as cluster centres and is inherently more robust to outliers.

**3. Assumption of Spherical, Equally-Sized Clusters**  
K-Means implicitly assumes that clusters are convex, isotropic (spherical in feature space), and of roughly equal size and density. Real customer data frequently violates these assumptions — for example, a high-income segment may have far fewer members than a budget segment, or customer groups may be elongated or crescent-shaped.

*Impact:* Non-spherical or unequal-density clusters are incorrectly split or merged, producing misleading segment boundaries.  
*Mitigation:* Visualise data before clustering; consider **Gaussian Mixture Models (GMM)** for ellipsoidal clusters (soft assignments, flexible covariance), or **DBSCAN** for arbitrary shapes and automatic outlier detection.

**Summary Table**

| Limitation | Effect on Segmentation | Mitigation |
|---|---|---|
| Poor initialisation | Inconsistent, locally optimal clusters | K-Means++, multiple restarts (`n_init`) |
| Outlier sensitivity | Distorted centroids, meaningless clusters | Outlier removal, RobustScaler, K-Medoids |
| Spherical cluster assumption | Wrong boundaries for non-spherical groups | GMM, DBSCAN, feature engineering / PCA |
| Fixed k requirement | Requires domain knowledge to set k | Elbow + Silhouette analysis, BIC for GMM |